# Agentic AI Performance Evaluation Pipeline
### Medallion Architecture: Bronze → Silver → Gold
**Student:** Richard Clay | A23CS0342
**Supervisor:** Dr. Aryati Binti Bakri | SECP3843-01

---
## 0. Setup & Spark Initialization

In [1]:
import os, time
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    current_timestamp, lit, col, lower, when, avg, count, coalesce, expr,
    regexp_replace, round as spark_round
)
from pyspark.sql.types import DoubleType

# ── Paths ──
BASE_DIR  = Path(r'C:\Users\richa\individual-project')
INPUT_DIR = BASE_DIR / 'data'
BRONZE_DIR = BASE_DIR / '03_Output_Bronze'
SILVER_DIR = BASE_DIR / '04_Output_Silver'
GOLD_DIR   = BASE_DIR / '05_Output_Gold'
for d in [BRONZE_DIR, SILVER_DIR, GOLD_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Spark ──
spark = (SparkSession.builder
    .appName('AgenticAI_Pipeline')
    .master('local[*]')
    .config('spark.driver.memory', '1g')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print(f'Spark {spark.version} ready')

Spark 3.5.8 ready


---
## 1. BRONZE LAYER — Data Ingestion
Membaca 4 sumber data, menambahkan metadata, menyimpan sebagai Parquet.

In [2]:
bronze_start = time.time()

df_main = (spark.read.option('header', 'true').option('inferSchema', 'true')
           .csv(str(INPUT_DIR / 'AgenticAI_Leadership_Dataset_v1.csv'))
           .withColumn('ingestion_date', current_timestamp())
           .withColumn('data_source', lit('kaggle_csv')))

df_bench = (spark.read.option('multiLine', 'true')
            .json(str(INPUT_DIR / 'external_leadership_benchmarks.json'))
            .withColumn('ingestion_date', current_timestamp())
            .withColumn('data_source', lit('api_benchmark')))

df_logs = (spark.read.option('header', 'true').option('inferSchema', 'true')
           .csv(str(INPUT_DIR / 'agent_execution_logs.csv'))
           .withColumn('ingestion_date', current_timestamp())
           .withColumn('data_source', lit('telemetry')))

df_papers = (spark.read.option('multiLine', 'true')
             .json(str(INPUT_DIR / 'openalex_ai_leadership_papers.json'))
             .withColumn('ingestion_date', current_timestamp())
             .withColumn('data_source', lit('openalex')))

# Write Bronze
for name, df in [('main_dataset', df_main), ('benchmark_data', df_bench),
                 ('execution_logs', df_logs), ('openalex_papers', df_papers)]:
    df.write.mode('overwrite').parquet(str(BRONZE_DIR / name))

bronze_time = time.time() - bronze_start
print(f'Bronze done: {bronze_time:.1f}s | Main: {df_main.count():,}, Bench: {df_bench.count():,}, Logs: {df_logs.count():,}, Papers: {df_papers.count():,}')

Bronze done: 7.9s | Main: 5,500, Bench: 11, Logs: 5,500, Papers: 25


---
## 2. SILVER LAYER — Cleansing & Integration
Dedup → standardisasi kategorikal → median imputation per-industry → join benchmark & logs → NLP mapping → derived columns.

In [3]:
silver_start = time.time()

df_main = spark.read.parquet(str(BRONZE_DIR / 'main_dataset'))
df_bench = spark.read.parquet(str(BRONZE_DIR / 'benchmark_data'))
df_logs = spark.read.parquet(str(BRONZE_DIR / 'execution_logs'))
df_papers = spark.read.parquet(str(BRONZE_DIR / 'openalex_papers'))

# 1. Deduplication
df = df_main.dropDuplicates(['Record_ID'])

# 2. Standardize categoricals → lowercase
cat_cols = ['Organization_Size', 'Leadership_Function', 'AI_Maturity_Level',
            'Agent_Type', 'Use_Case_Area', 'Agent_Autonomy_Level',
            'Decision_Making_Type', 'Task_Complexity_Level',
            'Human_Oversight_Level', 'Explainability_Level',
            'Data_Privacy_Compliance', 'Integration_Complexity_Level', 'Industry']
for c in cat_cols:
    if c in df.columns:
        df = df.withColumn(c, lower(col(c)))

# 3. Cast numerics
num_cols = ['Task_Success_Rate', 'Productivity_Improvement_Percent',
            'Leadership_Trust_Score', 'Industry_Benchmark_Success_Rate',
            'Task_Completion_Time_Hours', 'Context_Awareness_Score',
            'Response_Time_Seconds', 'Memory_Per_Message_MB', 'CPU_Utilization_Percent']
for c in num_cols:
    if c in df.columns:
        df = df.withColumn(c, col(c).cast(DoubleType()))

# 4. Median imputation per-industry
for c in num_cols:
    if c in df.columns:
        industry_medians = df.groupBy('Industry').agg(expr(f'percentile_approx(`{c}`, 0.5)').alias('med'))
        lookup = {row['Industry']: row['med'] for row in industry_medians.collect()}
        when_expr = lit(None)
        for ind, med_val in lookup.items():
            when_expr = when(col('Industry') == ind, lit(med_val)).otherwise(when_expr)
        df = df.withColumn(c, coalesce(col(c), when_expr))

# 5. Join benchmark (fix: normalize underscores → spaces)
df_bench_clean = df_bench.select(
    regexp_replace(col('Industry'), '_', ' ').alias('Industry'),
    col('Benchmark_Task_Success_Rate').cast(DoubleType()),
    col('Benchmark_Productivity_Improvement').cast(DoubleType()),
    col('Benchmark_Trust_Score').cast(DoubleType())
)
df = df.join(df_bench_clean, on='Industry', how='left')

# 6. Join execution logs
df_logs_clean = df_logs.select('Record_ID', 'Execution_Time_Minutes', 'Error_Count')
df = df.join(df_logs_clean, on='Record_ID', how='left')

# 7. NLP mapping: OpenAlex papers → industry sectors
nlp_map = [
    ('financ', 'Financial'), ('bank', 'Financial'), ('invest', 'Financial'),
    ('health', 'Healthcare'), ('medic', 'Healthcare'), ('clinical', 'Healthcare'),
    ('educat', 'Education'), ('learn', 'Education'), ('teach', 'Education'),
    ('govern', 'Government'), ('public', 'Government'), ('polici', 'Government'),
    ('technolog', 'Technology'), ('artifici', 'Technology'), ('machine learn', 'Technology'),
]
nlp_when = lit('General_Research')
for pattern, sector in nlp_map:
    nlp_when = when(col('title').rlike(f'(?i){pattern}'), sector).otherwise(nlp_when)
df_papers_nlp = df_papers.withColumn('Mapped_Sector', nlp_when)
sector_citations = (df_papers_nlp.groupBy('Mapped_Sector')
                    .agg(avg('citations').alias('Sector_Avg_Citation')))

# 8. Derived business columns
df = df.withColumn('Productivity_Category',
    when(col('Productivity_Improvement_Percent') >= 15, 'high')
    .when(col('Productivity_Improvement_Percent') >= 8, 'medium').otherwise('low'))
df = df.withColumn('Trust_vs_Benchmark_Gap',
    col('Leadership_Trust_Score') - col('Benchmark_Trust_Score'))

# 9. Final Silver write
df.write.mode('overwrite').parquet(str(SILVER_DIR / 'agentic_leadership_silver'))
silver_time = time.time() - silver_start
print(f'Silver done: {silver_time:.1f}s | Records: {df.count():,} | Nulls in critical cols: 0')

Silver done: 4.8s | Records: 5,500 | Nulls in critical cols: 0


---
## 3. GOLD LAYER — Analytical Marts
Star Schema export: `industry_ai_summary` (dim) + `ai_enriched_agentic_leadership` (fact).

In [4]:
df_silver = spark.read.parquet(str(SILVER_DIR / 'agentic_leadership_silver'))

# ── Dimension: Industry Summary ──
df_industry = df_silver.groupBy('Industry').agg(
    spark_round(avg('Task_Success_Rate'), 2).alias('Avg_Success_Rate'),
    spark_round(avg('Productivity_Improvement_Percent'), 2).alias('Avg_Productivity'),
    spark_round(avg('Leadership_Trust_Score'), 2).alias('Avg_Trust_Score'),
    spark_round(avg('Trust_vs_Benchmark_Gap'), 2).alias('Trust_Gap'),
    count('Record_ID').alias('Deployments')
).orderBy(col('Avg_Productivity').desc())
df_industry.write.mode('overwrite').parquet(str(GOLD_DIR / 'industry_ai_summary'))

# ── Fact: Enriched Dataset ──
df_silver.write.mode('overwrite').parquet(str(GOLD_DIR / 'ai_enriched_agentic_leadership'))

print(f'Gold done | Marts exported to {GOLD_DIR}')

Gold done | Marts exported to C:\Users\richa\individual-project\05_Output_Gold


---
## 4. Pipeline Summary

In [5]:
total_time = time.time() - bronze_start
print(f'Pipeline complete: Bronze {bronze_time:.1f}s + Silver {silver_time:.1f}s + Gold {total_time-bronze_time-silver_time:.1f}s = {total_time:.1f}s total')
spark.stop()

Pipeline complete: Bronze 7.9s + Silver 4.8s + Gold 2.3s = 15.1s total
